# EDA — VLearn AI Tutor: tìm pain point có bằng chứng

**Đơn vị phân tích:** một `turn_id` = một câu hỏi của học viên + một câu trả lời của tutor.  
**Mục tiêu:** tìm pattern có số đếm, lấy mẫu để đọc tay, rồi hình thành pain statement.  
**Quan trọng:** các `pain_*` bên dưới chỉ là heuristic sàng lọc. Muốn kết luận tutor trả lời sai kiến thức/citation sai cần đối chiếu tài liệu nguồn.

In [ ]:
from __future__ import annotations

import ast
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_colwidth", 160)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42


## 1. Nạp dữ liệu an toàn và kiểm tra schema

Không hardcode đường dẫn tuyệt đối của một thành viên. Có thể override bằng `VLEARN_CHATLOG_CSV`.

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa pyproject.toml")


PROJECT_ROOT = find_project_root()
default_csv = PROJECT_ROOT / "data/vlearn-pack/chatlog/chat_history_anonymized_for_hackathon.csv"
CSV_PATH = Path(os.getenv("VLEARN_CHATLOG_CSV", default_csv)).expanduser().resolve()
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy data: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print(f"Data: {CSV_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head(3))


In [ ]:
required = {
    "conversation_id", "user_id", "day_code", "turn_id", "message_id",
    "role", "content", "citations", "rating", "asked_check_question",
    "message_created_at", "llm_call_count", "models_used",
    "total_input_tokens", "total_output_tokens", "avg_latency_ms",
}
missing = required - set(df.columns)
assert not missing, f"Thiếu cột bắt buộc: {sorted(missing)}"

df["message_created_at"] = pd.to_datetime(df["message_created_at"], utc=True, errors="coerce")
for col in ["llm_call_count", "total_input_tokens", "total_output_tokens", "avg_latency_ms"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

overview = pd.Series({
    "messages": len(df),
    "turns": df["turn_id"].nunique(),
    "conversations": df["conversation_id"].nunique(),
    "users": df["user_id"].nunique(),
    "days": df["message_created_at"].dt.date.nunique(),
    "duplicate_message_id": df["message_id"].duplicated().sum(),
    "missing_content": df["content"].isna().sum(),
})
display(overview.to_frame("value"))
display(df.isna().mean().sort_values(ascending=False).rename("missing_rate").to_frame())


## 2. Ghép câu hỏi và câu trả lời theo turn

Cell này kiểm tra mỗi turn có đúng một message `student` và một message `tutor`, rồi tạo bảng `turns` để mọi tỷ lệ dùng cùng mẫu số.

In [ ]:
role_counts = df.groupby(["turn_id", "role"]).size().unstack(fill_value=0)
display(role_counts.value_counts().rename("number_of_turns").to_frame())

student = (
    df.loc[df["role"].eq("student"), ["turn_id", "content", "message_id"]]
    .rename(columns={"content": "student_text", "message_id": "student_message_id"})
)
tutor_cols = [
    "turn_id", "content", "message_id", "move_used", "citations", "misconceptions",
    "follow_ups", "rating", "asked_check_question", "conversation_id", "user_id",
    "day_code", "message_created_at", "llm_call_count", "models_used",
    "total_input_tokens", "total_output_tokens", "avg_latency_ms",
]
tutor = df.loc[df["role"].eq("tutor"), tutor_cols].rename(
    columns={"content": "tutor_text", "message_id": "tutor_message_id"}
)
turns = tutor.merge(student, on="turn_id", how="outer", validate="one_to_one")
assert turns["student_text"].notna().all() and turns["tutor_text"].notna().all()
print(f"Paired turns: {len(turns):,}")
display(turns[["turn_id", "student_text", "tutor_text", "rating"]].head(3))


## 3. Feature engineering có định nghĩa kiểm lại được

- `pain_no_citation`: danh sách citation rỗng.
- `pain_cannot_answer`: câu trả lời chứa cụm nói không tìm thấy/không đủ nguồn.
- `pain_overlong`: câu trả lời trên 180 từ **và** dài hơn câu hỏi ít nhất 8 lần.
- `pain_no_clarification`: câu hỏi rất ngắn/mơ hồ nhưng tutor không hỏi lại.
- `pain_slow`: latency từ p90 trở lên.
- `pain_downvote`: rating là `down`.

Ngưỡng được khai báo tập trung để người khác có thể thay đổi và chạy lại.

In [ ]:
def parse_json_list(value: object) -> list:
    if pd.isna(value) or value == "":
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = json.loads(str(value))
    except json.JSONDecodeError:
        parsed = ast.literal_eval(str(value))
    return parsed if isinstance(parsed, list) else []


def word_count(text: object) -> int:
    return len(re.findall(r"\w+", str(text), flags=re.UNICODE))


CANNOT_ANSWER_RE = re.compile(
    r"không (?:tìm thấy|có|đủ|thể xác định|thể trả lời)|"
    r"chưa (?:có|tìm thấy|đủ)|không có thông tin|ngoài (?:nội dung|tài liệu)",
    flags=re.IGNORECASE,
)
QUESTION_RE = re.compile(r"\?|bạn có thể|bạn muốn|vui lòng|hãy (?:cho|cung cấp)", re.IGNORECASE)
VAGUE_RE = re.compile(
    r"^(?:hi+|hello|chào|hả|gì|sao|tại sao|giải thích|tóm tắt|help|cứu)\W*$",
    flags=re.IGNORECASE,
)

turns["citation_list"] = turns["citations"].map(parse_json_list)
turns["student_words"] = turns["student_text"].map(word_count)
turns["tutor_words"] = turns["tutor_text"].map(word_count)
turns["response_to_question_ratio"] = turns["tutor_words"] / turns["student_words"].clip(lower=1)
turns["student_is_vague"] = (
    turns["student_words"].le(5)
    | turns["student_text"].str.strip().str.match(VAGUE_RE, na=False)
)
turns["tutor_asks_question"] = turns["tutor_text"].str.contains(QUESTION_RE, na=False)
latency_p90 = turns["avg_latency_ms"].quantile(0.90)

turns["pain_no_citation"] = turns["citation_list"].str.len().eq(0)
turns["pain_cannot_answer"] = turns["tutor_text"].str.contains(CANNOT_ANSWER_RE, na=False)
turns["pain_overlong"] = turns["tutor_words"].gt(180) & turns["response_to_question_ratio"].ge(8)
turns["pain_no_clarification"] = turns["student_is_vague"] & ~turns["tutor_asks_question"]
turns["pain_slow"] = turns["avg_latency_ms"].ge(latency_p90)
turns["pain_downvote"] = turns["rating"].eq("down")

pain_cols = [c for c in turns.columns if c.startswith("pain_")]
turns["pain_signal_count"] = turns[pain_cols].sum(axis=1)
print(f"p90 latency threshold: {latency_p90:,.0f} ms")


## 4. Pain signals: quy mô, tỷ lệ và phân khúc

Mỗi con số luôn báo cả tử số và mẫu số. `no_citation` không đồng nghĩa citation bắt buộc cho mọi câu; cần đọc mẫu theo intent trước khi chọn pain.

In [ ]:
pain_summary = pd.DataFrame({
    "turns_flagged": turns[pain_cols].sum(),
    "rate": turns[pain_cols].mean(),
}).sort_values("rate", ascending=False)
pain_summary["denominator"] = len(turns)
pain_summary["rate_pct"] = (pain_summary["rate"] * 100).round(1)
display(pain_summary[["turns_flagged", "denominator", "rate_pct"]])

ax = pain_summary.sort_values("rate_pct")["rate_pct"].plot.barh(figsize=(9, 5))
ax.set(title="Tỷ lệ turn bị gắn từng pain signal", xlabel="% tổng số turn", ylabel="")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)
plt.tight_layout()
plt.show()


In [ ]:
by_day = (
    turns.groupby("day_code", dropna=False)
    .agg(turns=("turn_id", "size"), users=("user_id", "nunique"), **{c: (c, "mean") for c in pain_cols})
    .sort_values("turns", ascending=False)
)
display(by_day.head(12).style.format({c: "{:.1%}" for c in pain_cols}))

rated = turns[turns["rating"].isin(["up", "down"])].copy()
print(f"Rated turns: {len(rated):,}/{len(turns):,} ({len(rated)/len(turns):.1%})")
if not rated.empty:
    display(pd.crosstab(rated["rating"], rated["pain_signal_count"], normalize="index").round(3))
    display(rated.groupby("rating")[["avg_latency_ms", "tutor_words", *pain_cols]].mean().round(2))


## 5. Hành vi sư phạm và hiệu năng

Phần này tìm pain về việc tutor lặp một kiểu phản hồi, ít kiểm tra hiểu bài, hoặc chậm. Không sử dụng `total_cost_usd` vì data dictionary cho biết cost tracking đang chưa hoạt động.

In [ ]:
display(turns["move_used"].value_counts(dropna=False).rename("turns").to_frame())
display(turns["asked_check_question"].value_counts(dropna=False).rename("turns").to_frame())
display(pd.DataFrame({
    "misconceptions_non_empty": turns["misconceptions"].map(parse_json_list).str.len().gt(0).value_counts(),
    "follow_ups_non_empty": turns["follow_ups"].map(parse_json_list).str.len().gt(0).value_counts(),
}).fillna(0).astype(int))

display(turns.groupby("models_used").agg(
    turns=("turn_id", "size"),
    median_latency_ms=("avg_latency_ms", "median"),
    p90_latency_ms=("avg_latency_ms", lambda s: s.quantile(0.90)),
    median_input_tokens=("total_input_tokens", "median"),
    median_output_tokens=("total_output_tokens", "median"),
).sort_values("turns", ascending=False).round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(turns["avg_latency_ms"], bins=40, ax=axes[0])
axes[0].axvline(latency_p90, color="red", linestyle="--", label="p90")
axes[0].set(title="Phân phối latency", xlabel="ms")
axes[0].legend()
sns.scatterplot(data=turns, x="total_input_tokens", y="avg_latency_ms", hue="models_used", alpha=.55, ax=axes[1])
axes[1].set(title="Input token và latency")
plt.tight_layout()
plt.show()


## 6. Lấy mẫu đọc tay — bước bắt buộc

Mỗi pain signal lấy tối đa 8 ví dụ theo seed cố định. Đọc và thêm cột `manual_valid`/`manual_note` trong bản làm việc nếu cần. Trích dẫn trong bài nộp chỉ dùng đoạn ngắn và ID ẩn danh.

In [ ]:
review_columns = [
    "turn_id", "conversation_id", "day_code", "rating", "student_text", "tutor_text",
    "citations", "avg_latency_ms", "student_words", "tutor_words", "pain_signal_count",
]

def sample_for_review(signal: str, n: int = 8) -> pd.DataFrame:
    pool = turns.loc[turns[signal], review_columns]
    if pool.empty:
        return pool
    return pool.sample(min(n, len(pool)), random_state=RANDOM_STATE).sort_values("turn_id")


for signal in pain_cols:
    print(f"\n### {signal}: {int(turns[signal].sum())}/{len(turns)} turns")
    display(sample_for_review(signal))


## 7. Xếp hạng pain candidate và chuyển thành evidence

Bảng dưới đây hỗ trợ thảo luận, không tự động chọn đề tài. `affected_users`, `affected_turns`, tần suất trên user bị ảnh hưởng và liên hệ với downvote là những đầu vào cho bảng impact. Với rating, phải ghi rõ mẫu rất nhỏ và có selection bias.

In [ ]:
rows = []
for signal in pain_cols:
    flagged = turns[turns[signal]]
    rated_flagged = flagged[flagged["rating"].isin(["up", "down"])]
    rows.append({
        "candidate": signal,
        "affected_turns": len(flagged),
        "affected_turn_rate": len(flagged) / len(turns),
        "affected_users": flagged["user_id"].nunique(),
        "affected_user_rate": flagged["user_id"].nunique() / turns["user_id"].nunique(),
        "turns_per_affected_user": len(flagged) / max(flagged["user_id"].nunique(), 1),
        "rated_n": len(rated_flagged),
        "downvote_rate_among_rated": rated_flagged["rating"].eq("down").mean(),
    })

candidate_summary = pd.DataFrame(rows).sort_values(
    ["affected_users", "affected_turns"], ascending=False
)
display(candidate_summary.style.format({
    "affected_turn_rate": "{:.1%}",
    "affected_user_rate": "{:.1%}",
    "turns_per_affected_user": "{:.2f}",
    "downvote_rate_among_rated": "{:.1%}",
}))


### Checklist kết luận pain point

Với mỗi candidate đáng chú ý:

1. Đọc tay ít nhất 20 mẫu flagged và 10 mẫu không flagged.
2. Ghi quy tắc phân loại, tử số, mẫu số và ít nhất 5 ví dụ nguyên văn ngắn.
3. Kiểm tra false positive; sửa regex/ngưỡng rồi chạy lại toàn bộ notebook.
4. Viết pain không có chữ AI: **ai – đang làm gì – vướng đâu – hậu quả gì**.
5. So sánh ít nhất 3 candidate bằng `người bị ảnh hưởng × tần suất × tổn thất/lần`.
6. Với lỗi đúng/sai hoặc citation, đối chiếu transcript/slide nguồn; EDA text không đủ để kết luận.

Gợi ý pain cần xác minh từ schema hiện tại: tutor thiếu grounding/citation; không trả lời được dù học viên đã chọn trang; phản hồi quá dài so với câu hỏi; không hỏi làm rõ input mơ hồ; hành vi sư phạm thiên lệch về `review_concept`; latency outlier.

## 8. Xuất bảng tổng hợp không chứa toàn bộ chatlog

Chỉ xuất số liệu aggregate. Thư mục `eda/outputs/` đã được gitignore để tránh vô tình commit dữ liệu dẫn xuất nhạy cảm.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "eda/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pain_summary.to_csv(OUTPUT_DIR / "pain_signal_summary.csv")
candidate_summary.to_csv(OUTPUT_DIR / "pain_candidate_summary.csv", index=False)
by_day.reset_index().to_csv(OUTPUT_DIR / "pain_signals_by_day_code.csv", index=False)
print(f"Đã ghi bảng aggregate vào: {OUTPUT_DIR}")
